# CSV File Ingestion with Pre-Generated Embeddings - Complete Lifecycle Management

This notebook demonstrates **end-to-end CSV file ingestion with pre-generated embeddings** using the modern **Ingestor fluent API** for efficient vector store operations.

## 🎯 **What This Notebook Accomplishes**
1. **Pre-Generated Embedding Support** - Process CSV files that already contain embedding vectors
2. **File-Embedding-Based Collection** - Direct embedding injection without runtime generation  
3. **Collection Lifecycle Management** - Create, update, search, and properly destroy collections
4. **Modern Ingestor Pipeline** - Using fluent API pattern following PDF ingestor best practices
5. **Complete Pipeline Demonstrations** - From file ingestion to searchable vector collections

## 📋 **FILE_EMBEDDING_BASED vs FILE_CONTENT_BASED**
- **FILE_CONTENT_BASED**: Generates embeddings at runtime using configured model
- **FILE_EMBEDDING_BASED**: Uses pre-generated embeddings from CSV column (faster, more efficient)

## 🔄 **Collection Management Strategy**
- **Cleanup First**: Destroy existing collections before creating new ones to avoid conflicts
- **Resource Management**: Properly clean up collections when no longer needed
- **Naming Convention**: Clear, descriptive collection names that indicate purpose and ingestor type

---

**🚀 Complete CSV document processing pipeline with pre-generated embeddings!**

<span style="color: #FFB000;">

## 1. Setup and Imports

Import required libraries for Ingestor workflow and CSV handling with pre-generated embeddings

</span>

In [ ]:
# 1. Environment Setup and Auto-Reload Configuration
print("🔧 Setting up CSV file-based vector store environment (with embeddings)...")

import sys
import os
import tempfile
from pathlib import Path
import pandas as pd
from datetime import datetime


print("✅ Auto-reload configured - modules will refresh on code changes")
print("📂 Python paths added for development")

# Now import the modules that we want to auto-reload
print("📚 Importing TeradataGenAI modules...")

# Teradataml imports
from teradataml import create_context, remove_context

# TeradataGenAI file-based imports
import teradatagenai
from teradatagenai import (
    Collection, CollectionManager,
    LocalConfig, S3Config, AzureBlobConfig, GCPConfig,
    BasicIngestor, NVIngestor, UnstructuredIngestor,
    ExtractionSchema, ColumnInfo, HNSW, FLAT, SearchParams,
    Ingestor, CollectionType, TeradataAI
)

# Get the directory one level above the current notebook file (notebooks directory)
base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__))) if '__file__' in dir() else os.path.dirname(os.getcwd())

from teradatasqlalchemy.types import VARCHAR, INTEGER, CLOB, VECTOR

print("✅ All imports successful!")
print(f"📂 Base directory: {base_dir}")
print("📚 Environment ready for CSV file-based vector store creation with embeddings")
print("🔄 Modules will now auto-reload when you modify source code")

🔧 Setting up CSV file-based vector store environment (with embeddings)...
✅ Auto-reload configured - modules will refresh on code changes
📂 Python paths added for development
📚 Importing TeradataGenAI modules...
✅ All imports successful!
📂 Base directory: c:\Aanchal\office\git\NexusAI\notebooks
📚 Environment ready for CSV file-based vector store creation with embeddings
🔄 Modules will now auto-reload when you modify source code


<span style="color: #FFB000;">

## 2. Secure Credential Collection

Collect all credentials securely using getpass - NOT displayed in outputs

</span>

In [ ]:
# 2. Secure Credential Collection
print("🔐 Collecting credentials securely...")

import os
from getpass import getpass

# Database credentials
base_url = getpass("Enter TD Base URL (e.g., https://aiop-btms4.td.teradata.com): ")
os.environ['TD_HOST'] = getpass("Enter TD_HOST: ")
os.environ['TD_USERNAME'] = getpass("Enter TD_USERNAME: ")
os.environ['TD_PASSWORD'] = getpass("Enter TD_PASSWORD: ")
os.environ['TD_BASE_URL'] = base_url
os.environ['TD_AUTH_MECH'] = 'BASIC'
os.environ['NVIDIA_API_KEY'] = getpass("Enter NVIDIA_API_KEY: ")

# ML model/API credentials
embeddings_base_url = getpass("Enter Embeddings Base URL: ")
nvidia_api_key = os.environ['NVIDIA_API_KEY']

# AWS S3 credentials
aws_access_key_id = getpass("Enter AWS Access Key ID: ")
aws_secret_access_key = getpass("Enter AWS Secret Access Key: ")
aws_session_token = getpass("Enter AWS Session Token: ")
s3_bucket_name = getpass("Enter S3 Bucket Name: ")
s3_region_name = getpass("Enter S3 Region Name (e.g., us-west-2): ")

# Azure credentials
azure_account_name = getpass("Enter Azure Account Name: ")
azure_account_key = getpass("Enter Azure Account Key: ")
azure_container_name = getpass("Enter Azure Container Name: ")

# GCP credentials
gcp_bucket_name = getpass("Enter GCP Bucket Name: ")
gcp_project_id = getpass("Enter GCP Project ID: ")
gcp_client_email = getpass("Enter GCP Client Email: ")
print("Enter GCP Secret JSON (paste the full JSON, then press Enter):")
import json
gcp_secret_str = getpass("GCP Secret JSON: ")
gcp_secret = json.loads(gcp_secret_str)

# Store variables for later use
TD_HOST = os.environ['TD_HOST']
TD_USER = os.environ['TD_USERNAME']
TD_PASSWORD = os.environ['TD_PASSWORD']

print("✅ All credentials collected securely")
print(f"🌐 Host: {TD_HOST}")
print(f"👤 User: {TD_USER}")
print("🔒 Credentials stored securely (not displayed)")
print("☁️ Multi-cloud credentials configured for S3, Azure Blob, and GCP")

In [4]:
# 2. Database Connection Configuration
print("🔗 Configuring database connection...")

# Create database context
create_context()

print("✅ Database connection established")
print(f"🌐 Connected to: {os.environ['TD_HOST']}")
print(f"👤 User: {os.environ['TD_USERNAME']}")

# Verify service health
try:
    health = CollectionManager.health()
    print("✅ Collection service is healthy")
except Exception as e:
    print(f"ℹ️ Service status: {e}")

🔗 Configuring database connection...
Authentication token is generated and set for the session.
✅ Database connection established
🌐 Connected to: 10.27.178.11
👤 User: oaf
✅ Collection service is healthy


<span style="color: #FFB000;">

## 3. Configure CSV Data (WITH Pre-Generated Embeddings)

Setup CSV files WITH pre-generated embeddings for FILE-EMBEDDING-BASED ingestion

</span>

In [23]:
# 3. Configure Local CSV File Ingestion with Pre-Generated Embeddings
print("📁 Setting up Local CSV File Ingestion with Pre-Generated Embeddings...")

# Use CSV files that contain pre-generated embeddings
csv_file_path = os.path.join(base_dir, 'example-data', 'csv', 'elements_with_embeddings_rag.csv')
csv_file_path_extra = os.path.join(base_dir, 'example-data', 'csv', 'ml_concepts_with_embeddings.csv')

# Verify file exists
if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"CSV file not found at: {csv_file_path}")

print(f"✅ CSV file found: {csv_file_path}")

# Load and inspect the CSV structure
csv_data = pd.read_csv(csv_file_path)

print(f"✓ CSV file loaded successfully")
print(f"✓ Number of rows: {len(csv_data)}")
print(f"✓ Columns: {list(csv_data.columns)}")
print(f"✓ Sample row: {csv_data.head(1).to_dict('records')[0] if len(csv_data) > 0 else 'No data'}")

# Display sample for verification
print("\n📊 Sample data:")
print(csv_data.head())

# Define collection names with clear, descriptive identifiers  
csv_collection_name = "csv_elements_embedding_ingestor"

# Configure embedding model for similarity_search queries
# Note: FILE_EMBEDDING_BASED uses pre-generated embeddings in CSV, but similarity_search
# still requires an embedding model to convert the query text into a vector
csv_embedding_model = TeradataAI(
    api_type="nim",
    model_name="nvidia/llama-3.2-nv-embedqa-1b-v2",
    api_base=embeddings_base_url
)

# Create LocalConfig for CSV file from specified path
csv_local_config = LocalConfig(
    files=[csv_file_path],  # Use the specific file path
    files_type="csv",
    delimiter=",",  # Standard CSV delimiter
)

print(f"\n✅ LocalConfig created for CSV file ingestion with embeddings")
print(f"✓ File path: {csv_file_path}")
print(f"✓ File type: csv")
print(f"✓ Collection Type: FILE_EMBEDDING_BASED")
print(f"✓ Embedding model for queries: {csv_embedding_model.model_name}")
print(f"✓ Note: Embeddings are pre-generated in CSV (1536 dimensions)")

📁 Setting up Local CSV File Ingestion with Pre-Generated Embeddings...
✅ CSV file found: c:\Aanchal\office\git\NexusAI\notebooks\example-data\csv\elements_with_embeddings_rag.csv
✓ CSV file loaded successfully
✓ Number of rows: 45
✓ Columns: ['text', 'embeddings', 'metadata']
✓ Sample row: {'text': 'ResearchGate\n\nSee discussions, stats, and author profiles for this publication at: https://www.researchgate.net/publication/388414789\n\nRetrieval-Augmented Generation (RAG): Advancing AI with Dynamic Knowledge\n\nIntegration\n\nPreprint · January 2025 DOI: 10.13140/RG.2.2.30888.89606\n\nCITATIONS\n\nREADS\n\n0\n\n1,632\n\n1 author:\n\nDouglas C Youvan\n\nyouvan.ai\n\n4,515 PUBLICATIONS 6,347 CITATIONS\n\nSEE PROFILE\n\nAll content following this page was uploaded by Douglas C Youvan on 27 January 2025.\n\nThe user has requested enhancement of the downloaded file.\n\nThe user has requested enhancement of the downloaded file.', 'embeddings': '[-0.032472162112213, -0.007920537729032, 0.0214

<span style="color: #FFB000;">

## 4. Configure Extraction Schema for Pre-Generated Embeddings

Set up ExtractionSchema with embedding_columns parameter for pre-generated embedding ingestion

</span>

In [24]:
# 4. Configure Extraction Schema with Pre-Generated Embeddings
print("📋 Configuring extraction schema for CSV with embeddings...")

# Configure extraction schema for data with pre-generated embeddings
csv_extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(
            name="text",
            datatype=VARCHAR(100),
            description="Element or concept name"
        )
    ],
    embedding_columns=[
        ColumnInfo(
            name="embeddings",
            description="Pre-generated embedding vector (1536 dimensions)"
        )
    ]
)

print("✅ Extraction schema configured for CSV with pre-generated embeddings")
print(f"  Collection type: FILE_EMBEDDING_BASED")
print(f"  Data columns: element, description, category")
print(f"  Embedding columns: embedding (1536 dimensions)")
print(f"  Note: No runtime embedding generation needed!")

📋 Configuring extraction schema for CSV with embeddings...
✅ Extraction schema configured for CSV with pre-generated embeddings
  Collection type: FILE_EMBEDDING_BASED
  Data columns: element, description, category
  Embedding columns: embedding (1536 dimensions)
  Note: No runtime embedding generation needed!


---

## Test 1: FILE-EMBEDDING-BASED Ingestion

**Objective:** Test CSV file ingestion with pre-generated embeddings

**Expected Behavior:**
1. Upload CSV files → Vector Store
2. Service extracts structured data from CSV
3. Service uses pre-generated embeddings (no runtime generation)
4. Data loaded into tables with searchable content

**What Should Happen:**
- ✅ Files upload successfully
- ✅ CSV parsing extracts data
- ✅ Pre-generated embeddings used directly (faster)
- ✅ Data tables created with vector search capability

**Proof Point:** This workflow enables efficient semantic search without embedding generation overhead

In [ ]:
# 5. Create and Execute CSV File Ingestion Pipeline with Pre-Generated Embeddings
print("🚀 Creating CSV file ingestion pipeline with PRE-GENERATED EMBEDDINGS...")

# Clean up existing collection if it exists
try:
    existing_collection = Collection(name=csv_collection_name)
    existing_collection.destroy()
    print(f"✅ Cleaned up existing collection: {csv_collection_name}")
except Exception as e:
    print(f"ℹ️ No existing collection to clean up: {e}")

print("\n" + "="*80)
print("🔌 CSV FILE INGESTION WITH PRE-GENERATED EMBEDDINGS")
print("="*80)
print("\nUsing FILE_EMBEDDING_BASED - embeddings are in the CSV file")
print(f"Source file: {csv_file_path}")
print("-"*80)

try:
    # Build Ingestor pipeline using fluent API for pre-generated embeddings
    csv_ingestor_pipeline = (
        Ingestor(
            name=csv_collection_name,
            type=CollectionType.FILE_EMBEDDING_BASED,  # USE PRE-GENERATED EMBEDDINGS
            target_database=TD_USER,
            description="CSV data ingestion with pre-generated embeddings using Ingestor API",
            embedding_model=csv_embedding_model
        )
        .files(
            files=csv_local_config,
            extraction_schema=csv_extraction_schema
        )
        .upsert(indexing_algorithm=HNSW(
            metric="COSINE",
            ef_construction=200,
            num_connpernode=32))
    )
    
    print("✅ Ingestor pipeline created successfully")
    print(f"  Collection: {csv_collection_name}")
    print(f"  Type: FILE_EMBEDDING_BASED (pre-generated)")
    print(f"  File: {csv_file_path}")
    print(f"  Embedding Source: CSV column 'embedding' (1536 dims)")
    
    print("\n🚀 Executing CSV ingestion pipeline...")
    result = csv_ingestor_pipeline.run()
    
    success = result.get('status', {}).get('success', False)
    
    if success:
        print("\n✅ CSV INGESTION WITH EMBEDDINGS SUCCESS!")
        print(f"  Collection: {csv_collection_name}")
        print(f"  Status: CSV file processed with pre-generated embeddings")
            
    else:
        print(f"\n❌ CSV INGESTION FAILED")
        print(f"  Status: {result.get('status', {})}")
        
        try:
            collection_check = Collection(name=csv_collection_name)
            status_data = collection_check.status(return_type="json")
            print(f"\n  Collection Status Details:")
            print(f"    Status: {status_data.get('collection_status')}")
            if 'error_message' in status_data:
                print(f"    Error: {status_data['error_message']}")
        except Exception as status_error:
            print(f"  Could not get collection status: {status_error}")
        
except Exception as e:
    print(f"\n❌ INGESTION FAILED WITH EXCEPTION")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

🚀 Creating CSV file ingestion pipeline with PRE-GENERATED EMBEDDINGS...
Collection csv_elements_embedding_ingestor initialized for use.
Collection destroy request is accepted and in progress
✅ Cleaned up existing collection: csv_elements_embedding_ingestor

🔌 CSV FILE INGESTION WITH PRE-GENERATED EMBEDDINGS

Using FILE_EMBEDDING_BASED - embeddings are in the CSV file
Source file: c:\Aanchal\office\git\NexusAI\notebooks\example-data\csv\elements_with_embeddings_rag.csv
--------------------------------------------------------------------------------
✅ Ingestor pipeline created successfully
  Collection: csv_elements_embedding_ingestor
  Type: FILE_EMBEDDING_BASED (pre-generated)
  File: c:\Aanchal\office\git\NexusAI\notebooks\example-data\csv\elements_with_embeddings_rag.csv
  Embedding Source: CSV column 'embedding' (1536 dims)

🚀 Executing CSV ingestion pipeline...
Initializing collection pipeline...
Collection csv_elements_embedding_ingestor initialized for use.
Creating collection...

In [26]:
result

{'status': {'success': True,
  'errors': [],
  'warnings': ['No ingestor specified; BasicIngestor() will be used by default'],
  'message': 'Pipeline executed successfully'},
 'collection': 'csv_elements_embedding_ingestor',
 'operation_details': {'create_collection': {'success': True,
   'api_response': 'Collection initialized successfully',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_elements_embedding_ingestor',
     'collection_status': 'initialized'}}},
  'ingest': {'success': True,
   'api_response': 'Files uploaded successfully and ingestion in progress',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_elements_embedding_ingestor',
     'collection_status': 'initialized'},
    'warnings': ['Operation completed but success status unclear']}},
  'create_index': {'success': True,
   'api_response': None,
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_elements_embed

In [ ]:
# Test similarity search to verify ingestion worked
try:
    print("\n🔍 Verification: Testing similarity search on data with embeddings...")
    collection = Collection(name=csv_collection_name)
    
    # Test search for elements
    search_results = collection.similarity_search(
        question="What elements are available?",
        top_k=5
    )
    
    print("Search results for elements:")
    print(search_results)
        
except Exception as search_error:
    print(f"  ⚠️ Search verification failed: {search_error}")


🔍 Verification: Testing similarity search on data with embeddings...
Collection csv_elements_embedding_ingestor initialized for use.
Search results for elements:
similar_objects_count:5
similar_objects:
      score DataBaseName                                      TableName  TD_ID                       td_filename                                                                                                  text  index_label
0  0.003321          oaf  ingest_table_c94dd2ed353d424bbc3444da1e8dd9c9     42  elements_with_embeddings_rag.csv  10. References  The field of Retrieval-Augmented Generation (RAG) is an evolving area of artificial             2
1 -0.000153          oaf  ingest_table_c94dd2ed353d424bbc3444da1e8dd9c9     43  elements_with_embeddings_rag.csv  ResearchGate  See discussions, stats, and author profiles for this publication at: https://www.resea            4
2  0.001684          oaf  ingest_table_c94dd2ed353d424bbc3444da1e8dd9c9     23  elements_with_embeddings_rag.csv

In [ ]:
# Test search for specific concepts
try:
    search_results2 = collection.similarity_search(
        question="Machine learning concepts",
        top_k=3
    )
    
    print("Search results for ML concepts:")
    print(search_results2)
        
except Exception as search_error:
    print(f"  ⚠️ Search verification failed: {search_error}")

Search results for ML concepts:
similar_objects_count:3
similar_objects:
      score DataBaseName                                      TableName  TD_ID                       td_filename                                                                                                  text  index_label
0 -0.000398          oaf  ingest_table_c94dd2ed353d424bbc3444da1e8dd9c9     17  elements_with_embeddings_rag.csv  5. Challenges in Real-Time Performance Optimization:  o Despite its advantages, integrating retrieva            2
1 -0.000018          oaf  ingest_table_c94dd2ed353d424bbc3444da1e8dd9c9     39  elements_with_embeddings_rag.csv  8.3 Enhancing Retrieval Accuracy with Reinforcement Learning  Future RAG models will incorporate rei            1
2  0.000022          oaf  ingest_table_c94dd2ed353d424bbc3444da1e8dd9c9     22  elements_with_embeddings_rag.csv  1. Multimodal RAG Approaches:  o Research focusing on extending RAG beyond text, incorporating image            0)


---

## ☁️ Part 2: Multi-Cloud CSV Storage Integration with Pre-Generated Embeddings

**Use Case:** Ingest CSV files with pre-generated embeddings from cloud storage (S3, Azure Blob, GCP)  
**Key Feature:** Multi-cloud CSV integration with FILE_EMBEDDING_BASED for efficient vector search

### 📊 CSV Cloud Processing Capabilities with Pre-Generated Embeddings:
- **Cloud Storage**: S3Config, AzureBlobConfig, GCPConfig for CSV files with embeddings
- **Pre-Generated Embeddings**: Use existing 1536-dimensional vectors from CSV column
- **Efficient Processing**: Skip runtime embedding generation for faster ingestion
- **Collection Management**: Proper cleanup and lifecycle management for CSV collections
- **Search Capabilities**: Leverage pre-computed embeddings for similarity search

**🚀 Complete CSV cloud processing pipeline with pre-generated embeddings!**

In [29]:
# Configure S3 CSV File Ingestion with Pre-Generated Embeddings
print("📊 Setting up S3 CSV File Ingestion with Pre-Generated Embeddings...")

# Define CSV cloud storage collection names
s3_csv_collection_name = "csv_s3_elements_embedding"
azure_csv_collection_name = "csv_azure_elements_embedding"  
gcp_csv_collection_name = "csv_gcp_elements_embedding"

# S3 configuration for CSV file processing with credentials from getpass
s3_csv_config = S3Config(
    bucket=s3_bucket_name,
    key="csv-data/elements_with_embeddings_rag.csv",
    region_name=s3_region_name,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token,
    files_type="csv",
    delimiter=","
)

# Configure extraction schema for S3 CSV content with embeddings
s3_csv_extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(100), description="Element name")
    ],
    embedding_columns=[
        ColumnInfo(name="embeddings", description="Pre-generated embedding")
    ]
)

print("✅ S3 CSV configuration with embeddings complete!")
print(f"📂 S3 bucket: {s3_csv_config.bucket}")
print(f"📊 CSV file: {s3_csv_config.key}")
print(f"🌐 S3 region: {s3_csv_config.region_name}")
print("🔄 Ready for S3 CSV ingestion with pre-generated embeddings")

📊 Setting up S3 CSV File Ingestion with Pre-Generated Embeddings...
✅ S3 CSV configuration with embeddings complete!
📂 S3 bucket: genaitestaanchal
📊 CSV file: csv-data/elements_with_embeddings_rag.csv
🌐 S3 region: us-west-2
🔄 Ready for S3 CSV ingestion with pre-generated embeddings


In [30]:
# Clean up existing S3 CSV collection before creating new one
print(f"🧹 Cleaning up existing S3 CSV collection: {s3_csv_collection_name}")
try:
    Collection(name=s3_csv_collection_name).destroy()
    print("✅ Existing S3 CSV collection destroyed")
except Exception as e:
    print(f"ℹ️ No existing S3 CSV collection to destroy: {e}")

# Create S3 CSV Collection Pipeline with Pre-Generated Embeddings
print("🚀 Creating S3 CSV Collection with Pre-Generated Embeddings...")

try:
    s3_csv_pipeline = (
        Ingestor(
            name=s3_csv_collection_name,
            type=CollectionType.FILE_EMBEDDING_BASED,  # PRE-GENERATED EMBEDDINGS
            target_database=TD_USER,
            description="Demo: S3 CSV data ingestion with pre-generated embeddings",
            embedding_model=csv_embedding_model
        )
        .files(
            files=s3_csv_config,
            extraction_schema=s3_csv_extraction_schema
        )
        .upsert(
            indexing_algorithm=HNSW(
                metric="COSINE",
                ef_construction=200,
                num_connpernode=32)
        ).run()
    )
    
    print("✅ S3 CSV collection with embeddings created successfully!")
    print(f"🎯 Pipeline success: {s3_csv_pipeline['status']['success']}")
    
    if s3_csv_pipeline['status']['success']:
        s3_csv_collection = s3_csv_pipeline["collection"]
        print(f"📚 S3 CSV Collection name: {s3_csv_collection.name}")
        print("🎉 S3 CSV data processed with pre-generated embeddings!")
    else:
        print(f"⚠️ S3 CSV Pipeline issues: {s3_csv_pipeline['status']['errors']}")
    
except Exception as e:
    print(f"ℹ️ S3 CSV collection creation: {e}")
    s3_csv_collection = Collection(name=s3_csv_collection_name)

print("\n📊 S3 CSV Processing Complete:")
print("  1️⃣ Downloaded CSV from S3 bucket")
print("  2️⃣ Processed data with pre-generated embeddings (no runtime generation)")
print("  3️⃣ Created searchable index directly from embedding column")
print("  🎯 Efficient cloud CSV integration with pre-computed vectors!")

🧹 Cleaning up existing S3 CSV collection: csv_s3_elements_embedding
Collection csv_s3_elements_embedding initialized for use.
Collection destroy request is accepted and in progress
✅ Existing S3 CSV collection destroyed
🚀 Creating S3 CSV Collection with Pre-Generated Embeddings...
Initializing collection pipeline...
Collection csv_s3_elements_embedding initialized for use.
Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'csv_s3_elements_embedding' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿

In [31]:
s3_csv_pipeline

{'status': {'success': True,
  'errors': [],
  'warnings': ['No ingestor specified; BasicIngestor() will be used by default'],
  'message': 'Pipeline executed successfully'},
 'collection': 'csv_s3_elements_embedding',
 'operation_details': {'create_collection': {'success': True,
   'api_response': 'Collection initialized successfully',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_s3_elements_embedding',
     'collection_status': 'initialized'}}},
  'ingest': {'success': True,
   'api_response': "File ingestion from remote storage for collection 'csv_s3_elements_embedding' has been scheduled.",
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_s3_elements_embedding',
     'collection_status': 'initialized'},
    'warnings': ['Operation completed but success status unclear']}},
  'create_index': {'success': True,
   'api_response': None,
   'status_result': {'success': True,
    'status_response': {'collection_

## Azure Blob Storage CSV Integration with Pre-Generated Embeddings

In [32]:
# Clean up existing Azure CSV collection
print(f"🧹 Cleaning up existing Azure CSV collection: {azure_csv_collection_name}")
try:
    Collection(name=azure_csv_collection_name).destroy()
    print("✅ Existing Azure CSV collection destroyed")
except Exception as e:
    print(f"ℹ️ No existing Azure CSV collection to destroy: {e}")

# Create Azure Blob CSV Collection Pipeline with Pre-Generated Embeddings
print("🚀 Creating Azure Blob CSV Collection with Pre-Generated Embeddings...")

# Configure extraction schema for Azure CSV content with embeddings
azure_csv_extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(100), description="Element name"),
    ],
    embedding_columns=[
        ColumnInfo(name="embeddings", description="Pre-generated embedding")
    ]
)

# Configure Azure Blob storage with credentials from getpass
azure_csv_config = AzureBlobConfig(
    container=azure_container_name,
    blob_name="csv-data/elements_with_embeddings_rag.csv",
    account_name=azure_account_name,
    account_key=azure_account_key,
    files_type="csv",
    delimiter=","
)

try:
    azure_csv_pipeline = (
        Ingestor(
            name=azure_csv_collection_name,
            type=CollectionType.FILE_EMBEDDING_BASED,  # PRE-GENERATED EMBEDDINGS
            target_database=TD_USER,
            description="Demo: Azure Blob CSV data ingestion with pre-generated embeddings",
            embedding_model=csv_embedding_model
        )
        .files(
            files=azure_csv_config,
            extraction_schema=azure_csv_extraction_schema
        )
        .upsert(
            indexing_algorithm=HNSW(
                metric="COSINE",
                ef_construction=200,
                num_connpernode=32)
        ).run()
    )
    
    print("✅ Azure Blob CSV collection with embeddings created successfully!")
    print(f"🎯 Pipeline success: {azure_csv_pipeline['status']['success']}")
    
    if azure_csv_pipeline['status']['success']:
        azure_csv_collection = azure_csv_pipeline["collection"]
        print(f"📚 Azure CSV Collection name: {azure_csv_collection.name}")
        print("🎉 Azure Blob CSV data processed with pre-generated embeddings!")
    else:
        print(f"⚠️ Azure CSV Pipeline issues: {azure_csv_pipeline['status']['errors']}")
    
except Exception as e:
    print(f"ℹ️ Azure CSV collection creation: {e}")
    azure_csv_collection = Collection(name=azure_csv_collection_name)

print("\n📊 Azure CSV Processing Complete:")
print("  1️⃣ Downloaded CSV from Azure Blob storage")
print("  2️⃣ Processed data with pre-generated embeddings")
print("  🎯 Multi-cloud CSV processing with Azure and pre-computed vectors!")

🧹 Cleaning up existing Azure CSV collection: csv_azure_elements_embedding
Collection csv_azure_elements_embedding initialized for use.
Collection destroy request is accepted and in progress
✅ Existing Azure CSV collection destroyed
🚀 Creating Azure Blob CSV Collection with Pre-Generated Embeddings...
Initializing collection pipeline...
Collection csv_azure_elements_embedding initialized for use.
Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'csv_azure_elements_embedding' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿

In [33]:
azure_csv_pipeline

{'status': {'success': True,
  'errors': [],
  'warnings': ['No ingestor specified; BasicIngestor() will be used by default'],
  'message': 'Pipeline executed successfully'},
 'collection': 'csv_azure_elements_embedding',
 'operation_details': {'create_collection': {'success': True,
   'api_response': 'Collection initialized successfully',
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_azure_elements_embedding',
     'collection_status': 'initialized'}}},
  'ingest': {'success': True,
   'api_response': "File ingestion from remote storage for collection 'csv_azure_elements_embedding' has been scheduled.",
   'status_result': {'success': True,
    'status_response': {'collection_name': 'csv_azure_elements_embedding',
     'collection_status': 'initialized'},
    'warnings': ['Operation completed but success status unclear']}},
  'create_index': {'success': True,
   'api_response': None,
   'status_result': {'success': True,
    'status_response': {

## Google Cloud Storage CSV Integration with Pre-Generated Embeddings

In [34]:
# Configure Google Cloud Storage CSV Integration with Pre-Generated Embeddings
print("☁️ Setting up Google Cloud Storage CSV configuration with Pre-Generated Embeddings...")

# Clean up existing GCP CSV collection
print(f"🧹 Cleaning up existing GCP CSV collection: {gcp_csv_collection_name}")
try:
    Collection(name=gcp_csv_collection_name).destroy()
    print("✅ Existing GCP CSV collection destroyed")
except Exception as e:
    print(f"ℹ️ No existing GCP CSV collection to destroy: {e}")

# Configure GCP storage with credentials from getpass
gcp_csv_config = GCPConfig(
    files_type="csv",
    bucket=gcp_bucket_name,
    blob_name="csv-data/elements_with_embeddings_rag.csv",
    project_id=gcp_project_id,
    delimiter=",",
    secret=gcp_secret
)

# Configure extraction schema for GCP CSV content with embeddings
gcp_csv_extraction_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(100), description="Element name"),
    ],
    embedding_columns=[
        ColumnInfo(name="embeddings", description="Pre-generated embedding")
    ]
)

print("✅ Google Cloud Storage CSV configuration with embeddings complete!")
print(f"📂 GCP bucket: {gcp_csv_config.bucket}")
print(f"📊 CSV file: {gcp_csv_config.blob_name}")
print("🔄 Ready for GCP CSV data ingestion with pre-generated embeddings")
print(f"🏢 Project: {gcp_csv_config.project_id}")

☁️ Setting up Google Cloud Storage CSV configuration with Pre-Generated Embeddings...
🧹 Cleaning up existing GCP CSV collection: csv_gcp_elements_embedding
Collection csv_gcp_elements_embedding initialized for use.
Collection destroy request is accepted and in progress
✅ Existing GCP CSV collection destroyed
✅ Google Cloud Storage CSV configuration with embeddings complete!
📂 GCP bucket: ak250090-filestore
📊 CSV file: csv-data/elements_with_embeddings_rag.csv
🔄 Ready for GCP CSV data ingestion with pre-generated embeddings
🏢 Project: icgcp-vector-store


In [35]:
# Create Google Cloud Storage CSV Collection Pipeline with Pre-Generated Embeddings
print("🚀 Creating Google Cloud Storage CSV Collection with Pre-Generated Embeddings...")

try:
    gcp_csv_pipeline = (
        Ingestor(
            name=gcp_csv_collection_name,
            type=CollectionType.FILE_EMBEDDING_BASED,  # PRE-GENERATED EMBEDDINGS
            target_database=TD_USER,
            description="Demo: GCP Storage CSV data ingestion with pre-generated embeddings",
            embedding_model=csv_embedding_model
        )
        .files(
            files=gcp_csv_config,
            extraction_schema=gcp_csv_extraction_schema
        )
        .upsert(
            indexing_algorithm=HNSW(
                metric="COSINE",
                ef_construction=200,
                num_connpernode=32)
        ).run()
    )
    
    print("✅ GCP Storage CSV collection with embeddings created successfully!")
    print(f"🎯 Pipeline success: {gcp_csv_pipeline['status']['success']}")
    
    if gcp_csv_pipeline['status']['success']:
        gcp_csv_collection = gcp_csv_pipeline["collection"]
        print(f"📚 GCP CSV Collection name: {gcp_csv_collection.name}")
        print("🎉 GCP Storage CSV data processed with pre-generated embeddings!")
    else:
        print(f"⚠️ GCP CSV Pipeline issues: {gcp_csv_pipeline['status']['errors']}")
    
except Exception as e:
    print(f"ℹ️ GCP CSV collection creation: {e}")
    gcp_csv_collection = Collection(name=gcp_csv_collection_name)

print("\n📊 GCP CSV Processing Complete:")
print("  1️⃣ Downloaded CSV from Google Cloud Storage")
print("  2️⃣ Processed data with pre-generated embeddings")
print("  3️⃣ Created searchable index from embedding column")
print("  🎯 Complete multi-cloud CSV processing pipeline with all three providers!")


🚀 Creating Google Cloud Storage CSV Collection with Pre-Generated Embeddings...
Initializing collection pipeline...
Collection csv_gcp_elements_embedding initialized for use.
Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
File ingestion from remote storage for collection 'csv_gcp_elements_embedding' has been scheduled.
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and 

In [36]:
# Comprehensive Multi-Cloud CSV Testing and Validation
print("\n🔍 Testing All CSV Collections with Pre-Generated Embeddings Across Multi-Cloud Infrastructure...")

# Define all CSV collections for comprehensive testing
all_csv_collections = [
    (s3_csv_collection_name, "AWS S3"),
    (azure_csv_collection_name, "Azure Blob"),
    (gcp_csv_collection_name, "Google Cloud")
]

# Queries for testing embedding-based search
test_queries = [
    "What chemical elements are available?",
    "Show me machine learning concepts",
    "Find data science topics",
    "What algorithms are included?"
]

print("\n📊 Multi-Cloud CSV Testing with Pre-Generated Embeddings:")
print("=" * 60)

for collection_name, cloud_provider in all_csv_collections:
    print(f"\n🌐 Testing {cloud_provider} CSV Collection: {collection_name}")
    print("-" * 50)
    
    try:
        collection_obj = Collection(name=collection_name)
        status = collection_obj.status(return_type="json")
        print(f"   ✅ Collection Status: {status.get('status', 'Unknown')}")
        
        for query in test_queries[:2]:
            try:
                search_result = collection_obj.similarity_search(
                    question=query,
                    search_params=SearchParams(top_k=2),
                    embedding_model=csv_embedding_model
                )
                print(f"   🎯 Query: '{query}' - Success")
                print(f"result{search_result}")
                
            except Exception as e:
                print(f"   ⚠️ Query: '{query}' - {str(e)[:50]}...")
                
    except Exception as e:
        print(f"   ❌ Collection Error: {str(e)[:50]}...")

print(f"\n🎯 MULTI-CLOUD CSV INTEGRATION WITH PRE-GENERATED EMBEDDINGS SUMMARY:")
print("✅ S3, Azure Blob, and GCP CSV collections created")
print("✅ FILE_EMBEDDING_BASED processing (no runtime embedding generation)")
print("✅ HNSW indexing for fast similarity search")
print("✅ Pre-computed vectors used directly")
print("✅ Complete cloud provider coverage for CSV ingestion")
print("\n🔄 All CSV collections ready for efficient vector search!")


🔍 Testing All CSV Collections with Pre-Generated Embeddings Across Multi-Cloud Infrastructure...

📊 Multi-Cloud CSV Testing with Pre-Generated Embeddings:

🌐 Testing AWS S3 CSV Collection: csv_s3_elements_embedding
--------------------------------------------------
Collection csv_s3_elements_embedding initialized for use.
   ✅ Collection Status: Unknown
   🎯 Query: 'What chemical elements are available?' - Success
resultsimilar_objects_count:2
similar_objects:
      score DataBaseName                                      TableName  TD_ID                       td_filename                                                                                                  text  index_label
0  0.010841          oaf  ingest_table_83721646cf8c4caea58aa696f916cb5f     33  elements_with_embeddings_rag.csv  2. Knowledge Retrieval:  o The query is matched against external knowledge repositories using search            1
1  0.014075          oaf  ingest_table_83721646cf8c4caea58aa696f916cb5f     40

## Section 8: Quick Document Management Demo (from_documents/add_documents/delete_documents)

Simple example demonstrating Collection.from_documents(), add_documents(), and delete_documents() with CSV data containing pre-generated embeddings.

In [37]:
# 8.1: Setup - Document Management Demo with Pre-Generated Embeddings
print("📊 CSV Document Management Scenario with Pre-Generated Embeddings\n")

doc_mgmt_collection = "csv_embedding_doc_mgmt_demo"

# Clean up existing collection
try:
    Collection(name=doc_mgmt_collection).destroy()
except:
    pass

# Setup extraction schema with embedding columns
doc_schema = ExtractionSchema(
    data_columns=[
        ColumnInfo(name="text", datatype=VARCHAR(100)),
    ],
    embedding_columns=[
        ColumnInfo(name="embeddings")
    ]
)

print("✅ Setup complete - Ready for document management operations with embeddings\n")

📊 CSV Document Management Scenario with Pre-Generated Embeddings



C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection does not exist or it has been destroyed successfully.
✅ Setup complete - Ready for document management operations with embeddings



In [43]:
# 8.2: Create Collection using from_documents() with Pre-Generated Embeddings
print("1️⃣ Creating collection with from_documents() (FILE_EMBEDDING_BASED)...")
try:
    doc_collection = Collection.from_documents(
        name=doc_mgmt_collection,
        documents=LocalConfig(files=[csv_file_path], files_type="csv"),
        type=CollectionType.FILE_EMBEDDING_BASED,  # PRE-GENERATED EMBEDDINGS
        extraction_schema=doc_schema,
        target_database=TD_USER,
        description="CSV document management demo with pre-generated embeddings",
        embedding=csv_embedding_model
    )
    print("   ✅ Collection created\n")
except Exception as e:
    print(f"   ⚠️ Error: {e}\n")
    doc_collection = Collection(name=doc_mgmt_collection)

1️⃣ Creating collection with from_documents() (FILE_EMBEDDING_BASED)...
Initializing collection pipeline...


C:\Aanchal\office\git\NexusAI\teradatagenai\vector_store\collection.py:639: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Creating collection...
Collection initialized successfully
Starting create collection operation...
Create Collection completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100              
Create Collection completed successfully
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with custom settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/10

In [44]:
# 8.3: Add Documents using add_documents() with Pre-Generated Embeddings
print("2️⃣ Adding documents with add_documents()...")
try:
    doc_collection.add_documents(
        documents=LocalConfig(files=[csv_file_path_extra], files_type="csv"),
        extraction_schema=doc_schema,
        target_database=TD_USER
    )
    print("   ✅ Documents added\n")
except Exception as e:
    print(f"   ⚠️ Error: {e}\n")

2️⃣ Adding documents with add_documents()...
Initializing collection pipeline...
Collection csv_embedding_doc_mgmt_demo initialized for use.
Creating collection...
Processing files...
Files uploaded successfully and ingestion in progress
Starting ingest operation...
Ingest completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Creating index with default settings...
Updating collection with index configuration for file based VS...
Collection update request is accepted and in progress
Starting update operation...
Update completed.                                                                          
Completed: ｜⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿⫿｜ 100% - 100/100   
Update completed successfully
Pipeline completed successfully! Operations: create_collection, ingest, create_index
   ✅ Documents added



In [45]:
# 8.4: Test with Similarity Search
print("3️⃣ Testing with similarity search...")
try:
    results = doc_collection.similarity_search(
        question="Find machine learning algorithms",
        search_params=SearchParams(top_k=2)
    )
    print("   ✅ Search successful\n")
except Exception as e:
    print(f"   ⚠️ Error: {e}\n")

3️⃣ Testing with similarity search...
   ✅ Search successful



In [46]:
results

similar_objects_count:2
similar_objects:
      score DataBaseName                                      TableName  TD_ID                      td_filename                                                                                                  text  index_label
0  0.055071          oaf  ingest_table_1c7addcef76c46a68944fcea81337daf     48  ml_concepts_with_embeddings.csv  Reinforcement Learning Principles  Reinforcement learning trains agents to make sequential decisions            1
1  0.067975          oaf  ingest_table_1c7addcef76c46a68944fcea81337daf     65  ml_concepts_with_embeddings.csv  Deep Learning and Neural Networks  Deep learning uses artificial neural networks with multiple layer            0)

In [47]:
# 8.5: Delete Documents using delete_documents()
print("4️⃣ Deleting documents with delete_documents()...")
try:
    doc_collection.delete_documents(documents=["elements_with_embeddings_rag.csv"])
    print("   ✅ Documents deleted\n")
except Exception as e:
    print(f"   ⚠️ Error: {e}\n")

4️⃣ Deleting documents with delete_documents()...
Collection update request is accepted and in progress
   ✅ Documents deleted



In [ ]:
# 7. Comprehensive CSV Collection Cleanup (Local and Cloud)
print("🧹 Comprehensive CSV Collection Cleanup Management...")
print("🔄 This includes local and all cloud storage collections\n")

# List all CSV collections created in this notebook
all_csv_collections = [
    csv_collection_name,
    s3_csv_collection_name,
    azure_csv_collection_name,
    gcp_csv_collection_name,
    doc_mgmt_collection
]

cleanup_results = []

for collection_name in all_csv_collections:
    try:
        print(f"🗑️ Destroying: {collection_name}")
        Collection(name=collection_name).destroy()
        cleanup_results.append(f"✅ {collection_name} - Successfully destroyed")
    except Exception as e:
        cleanup_results.append(f"ℹ️ {collection_name} - {e}")

print("\n📊 CLEANUP SUMMARY:")
for result in cleanup_results:
    print(f"  {result}")

print(f"\n🎯 CSV EMBEDDING NOTEBOOK COMPLETE!")
print("📚 What we demonstrated:")
print("  ✅ Local CSV file processing with pre-generated embeddings")
print("  ✅ Multi-cloud CSV integration (S3, Azure Blob, GCP)")
print("  ✅ FILE_EMBEDDING_BASED collection type (efficient, no runtime embedding)")
print("  ✅ Proper collection lifecycle management (create → use → destroy)")
print("  ✅ Complete resource cleanup to prevent memory leaks")
print("\n🔄 All CSV collections destroyed - system ready for next workflow!")

🧹 Comprehensive CSV Collection Cleanup Management...
🔄 This includes local and all cloud storage collections

🗑️ Destroying: csv_elements_embedding_ingestor
Collection csv_elements_embedding_ingestor initialized for use.
Collection destroy request is accepted and in progress
🗑️ Destroying: csv_s3_elements_embedding
Collection csv_s3_elements_embedding initialized for use.
Collection destroy request is accepted and in progress
🗑️ Destroying: csv_azure_elements_embedding
Collection csv_azure_elements_embedding initialized for use.
Collection destroy request is accepted and in progress
🗑️ Destroying: csv_gcp_elements_embedding
Collection csv_gcp_elements_embedding initialized for use.
Collection destroy request is accepted and in progress
🗑️ Destroying: csv_embedding_doc_mgmt_demo
Collection csv_embedding_doc_mgmt_demo initialized for use.
Collection destroy request is accepted and in progress

📊 CLEANUP SUMMARY:
  ✅ csv_elements_embedding_ingestor - Successfully destroyed
  ✅ csv_s3_elem

: 